In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd

join_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(join_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Price Distribution')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:

# copy the dataset
df_clean = df.copy()


df_clean.drop(columns=['Order_ID'], inplace=True)

In [ ]:
# Task 2: Write your code here:
df_clean.isna().sum()

In [ ]:
# dropna for rows with missing values
df_clean.dropna(inplace=True)
df_clean.isna().sum()

In [ ]:
print(f"Shape before: {df.shape} \nShape After: {df_clean.shape}")

In [ ]:
# Task 3: Write your code here:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
print(df_clean.select_dtypes(include=["object"]).columns)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder

categorical  = df_clean[['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']]

encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(categorical)

print(encoded)

In [ ]:
categorical_encoded = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(categorical.columns))
categorical_encoded

In [ ]:
categorical_encoded['Distance_km'] = df_clean['Distance_km']
df_encoded = categorical_encoded.merge(df_clean.drop(columns=categorical))
df_encoded.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


# spliting the data before scaling to avoid data leakage

X = df_encoded.drop(columns='Delivery_Time', axis=1)
y = df_encoded['Delivery_Time']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Task 6: Write your code here:
# Unneeded

In [ ]:
# Task 1: Write your code here:
# Done on cell 18

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error as sklearn_mse

# Using Kfold >> the target is a continous value

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)


In [ ]:
mse_ls = []

for fold_idx, (train_index, test_index) in enumerate(kfold.split(X)):

  X_train_scaled, X_test_scaled = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model.fit(X_train_scaled, y_train)

  # Predict
  y_pred = model.predict(X_test_scaled)

  # Calculate metrics
  mse = sklearn_mse(y_test, y_pred)

  # Store results
  mse_ls.append(mse)

print(sum(mse_ls)/len(mse_ls))

In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 6))
plt.hist(y_pred, bins=30, edgecolor='black')
plt.title('Distribution of Predictions')
plt.xlabel('Predicted Emission')
plt.ylabel('Count')
plt.show()

In [ ]:
# Task Bonus: Write your code here: